In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Load preprocessed pharmaceutical dataset
dataset_path = (
    r"C:\xampp\htdocs\ai-drug-price\data\cleaned_data_set\cleaned.csv"
)
pharmaceutical_data = pd.read_csv(dataset_path)

# Define predictor features and target metric
categorical_predictors = [
    "Generic Name",
    "Strength",
    "Pack Size",
    "Name of the Manufacturer",
    "Dosage Description",
    "Use For",
]
response_variable = "Price"

X_matrix = pharmaceutical_data[categorical_predictors]
y_vector = pharmaceutical_data[response_variable]

# Partition dataset into training and testing subsets (80/20 split)
X_train_set, X_test_set, y_train_set, y_test_set = train_test_split(
    X_matrix, y_vector, test_size=0.20, random_state=42
)

print(f"Training feature matrix shape: {X_train_set.shape}")
print(f"Testing feature matrix shape:  {X_test_set.shape}")

# Construct feature transformer pipeline using One-Hot Encoding
feature_encoder = ColumnTransformer(
    transformers=[
        (
            "ohe_categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_predictors,
        )
    ]
)

# Instantiate Random Forest Ensemble Regressor
regressor_estimator = RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1
)

# Build integrated processing and modeling pipeline
estimation_pipeline = Pipeline(
    steps=[
        ("feature_preprocessing", feature_encoder),
        ("regressor_model", regressor_estimator),
    ]
)

# Fit regression pipeline on training partition
estimation_pipeline.fit(X_train_set, y_train_set)

# Serialize and export trained pipeline object
output_model_path = (
    r"C:\xampp\htdocs\ai-drug-price\models\medicine_price_model.pkl"
)
os.makedirs(os.path.dirname(output_model_path), exist_ok=True)
joblib.dump(estimation_pipeline, output_model_path)

print(f"Trained pipeline model saved to: {output_model_path}")

# Perform evaluation inference on test dataset
y_predictions = estimation_pipeline.predict(X_test_set)

# Calculate quantitative regression performance metrics
mae_score = mean_absolute_error(y_test_set, y_predictions)
mse_score = mean_squared_error(y_test_set, y_predictions)
rmse_score = np.sqrt(mse_score)
r2_metric = r2_score(y_test_set, y_predictions)

# Output evaluation metrics
print("\n--- Model Evaluation Summary ---")
print(f"Mean Absolute Error (MAE):     {mae_score:.2f}")
print(f"Mean Squared Error (MSE):      {mse_score:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_score:.2f}")
print(f"R-squared (R2 Score):          {r2_metric:.4f}")

Training feature matrix shape: (40158, 6)
Testing feature matrix shape:  (10040, 6)
Trained pipeline model saved to: C:\xampp\htdocs\ai-drug-price\models\medicine_price_model.pkl

--- Model Evaluation Summary ---
Mean Absolute Error (MAE):     7.45
Mean Squared Error (MSE):      1415.98
Root Mean Squared Error (RMSE): 37.63
R-squared (R2 Score):          0.8443
